In [1]:
import pandas as pd
import sys

sys.path.append('..')
from src.models.clustering import (
    build_duplicate_graph,
    build_clusters_from_graph,
    attach_clusters_to_records,
    add_canonical_names,
    compute_cluster_stats,
)

from src.models.canonicalization import (
    build_canonical_entities,
    add_entity_quality_flags,
    entity_summary,
)

In [2]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_parquet(
    "../data/processed/processed_dataset.parquet"
)

duplicate_pairs = pd.read_parquet(
    "../data/processed/duplicate_pairs_filtered_sample.parquet"
)

scored_pairs = pd.read_parquet(
    "../data/processed/scored_pairs_sample.parquet"
)

print(f"Full dataset rows: {len(df):,}")
print(f"Filtered duplicate pairs: {len(duplicate_pairs):,}")
print(f"Scored pairs: {len(scored_pairs):,}")

Full dataset rows: 3,653,581
Filtered duplicate pairs: 253,413
Scored pairs: 2,865,301


In [3]:
# ============================================================
# RESTORE SAMPLE RECORDS
# ============================================================

sample_indices = set(scored_pairs["idx1"]) | set(scored_pairs["idx2"])

df_sample = df.loc[list(sample_indices)].copy()

print(f"Sample records: {len(df_sample):,}")

Sample records: 89,133


In [4]:
# ============================================================
# BUILD GRAPH FROM FILTERED DUPLICATE PAIRS
# ============================================================

graph = build_duplicate_graph(
    predicted_pairs=duplicate_pairs,
    score_col="duplicate_score",
    threshold=0.95,
)

clusters = build_clusters_from_graph(graph)

print(f"Graph nodes: {graph.number_of_nodes():,}")
print(f"Graph edges: {graph.number_of_edges():,}")
print(f"Non-singleton clustered records: {len(clusters):,}")
print(f"Non-singleton clusters: {clusters['entity_id'].nunique():,}")

Graph nodes: 19,396
Graph edges: 253,413
Non-singleton clustered records: 19,396
Non-singleton clusters: 5,953


In [5]:
# ============================================================
# ATTACH ENTITY ID TO ALL SAMPLE RECORDS
# ============================================================

deduplicated_sample = attach_clusters_to_records(
    df=df_sample,
    clusters=clusters,
)

deduplicated_sample = add_canonical_names(
    deduplicated_sample,
    name_col="name_latin",
)

print(f"Deduplicated sample rows: {len(deduplicated_sample):,}")

display(
    deduplicated_sample[
        [
            "record_id",
            "party_name",
            "name_latin",
            "entity_id",
            "cluster_size",
            "canonical_name",
        ]
    ].head()
)

Deduplicated sample rows: 89,133


,record_id,party_name,name_latin,entity_id,cluster_size,canonical_name
0,mng_rec_3604c8299570c9b81987c24172904fce,Уламбаяр Мөнхбаяр,ulambayar monhbayar,single_2621448,1,ulambayar monhbayar
1,mng_rec_098d98e586d451875fa8583139040706,сүхбаатар оюунэрдэнэ,suhbaatar oyuunerdene,entity_00004174,3,suhbaatar oyuunerdene
2,kgz_rec_cea45e458fc1c18065819b25cb2c96ca,Курманбекова Рахат Сапарбаевна,kurmanbekova rahat saparbaevna,single_2097162,1,kurmanbekova rahat saparbaevna
3,kaz_rec_b2a7bdd43eb32a7b30b214ab5b675c59,БАЯХМЕТОВА САЛТАНАТ МУРАТОВНА,bayahmetova saltanat muratovna,single_1048589,1,bayahmetova saltanat muratovna
4,arm_rec_902a34f25303eab1e1b4693858ace39c,Լևոն Պետրոսյան,leuon petrosyan,entity_00002706,2,leuon petrosyan


In [6]:
# ============================================================
# BUILD CANONICAL ENTITIES FOR FULL SAMPLE
# ============================================================

canonical_entities = build_canonical_entities(
    deduplicated_sample
)

canonical_entities = add_entity_quality_flags(
    canonical_entities,
    large_cluster_threshold=20,
    max_unique_names=5,
    max_unique_countries=3,
)

entity_summary(canonical_entities)

CANONICAL ENTITY SUMMARY
Entities: 75,690

Cluster size stats:
count    75690.000000
mean         1.177606
std          2.547148
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        369.000000
Name: cluster_size, dtype: float64

Singletons:
is_singleton
1    69737
0     5953
Name: count, dtype: int64

Large clusters:
is_large_cluster
0    75636
1       54
Name: count, dtype: int64

Suspicious entities:
is_suspicious_entity
0    75690
Name: count, dtype: int64


In [7]:
# ============================================================
# FINAL EXPORT SUMMARY
# ============================================================

total_records = len(deduplicated_sample)
total_entities = canonical_entities["entity_id"].nunique()

singleton_entities = (
    canonical_entities["cluster_size"] == 1
).sum()

non_singleton_entities = (
    canonical_entities["cluster_size"] > 1
).sum()

records_in_duplicate_clusters = (
    canonical_entities.loc[
        canonical_entities["cluster_size"] > 1,
        "cluster_size",
    ].sum()
)

print("=" * 60)
print("FINAL DEDUPLICATED SAMPLE SUMMARY")
print("=" * 60)

print(f"Input sample records: {total_records:,}")
print(f"Final entities: {total_entities:,}")
print(f"Singleton entities: {singleton_entities:,}")
print(f"Non-singleton entities: {non_singleton_entities:,}")
print(f"Records in duplicate clusters: {records_in_duplicate_clusters:,}")

print(
    f"Deduplication compression: "
    f"{1 - total_entities / total_records:.2%}"
)

FINAL DEDUPLICATED SAMPLE SUMMARY
Input sample records: 89,133
Final entities: 75,690
Singleton entities: 69,737
Non-singleton entities: 5,953
Records in duplicate clusters: 19,396
Deduplication compression: 15.08%


In [8]:
# ============================================================
# TOP LARGEST CANONICAL ENTITIES
# ============================================================

display(
    canonical_entities[
        [
            "entity_id",
            "canonical_name",
            "cluster_size",
            "canonical_country",
            "unique_names_count",
            "unique_countries_count",
            "unique_companies_count",
            "is_suspicious_entity",
        ]
    ]
    .sort_values("cluster_size", ascending=False)
    .head(30)
)

,entity_id,canonical_name,cluster_size,canonical_country,unique_names_count,unique_countries_count,unique_companies_count,is_suspicious_entity
3316,entity_00003316,neo metals holding limited,369,arm,1,1,3,0
4228,entity_00004228,svetlana ershova,274,arm,1,1,3,0
3091,entity_00003091,mirzaaziz mousakhanov,274,arm,1,1,1,0
439,entity_00000439,ardyounaberakan ynkeroutyoun by,230,arm,1,1,5,0
3585,entity_00003585,promishlennaya kompania,173,arm,1,1,5,0
3314,entity_00003314,neo metals holding limited bazhnetomserov sahm...,104,arm,1,1,2,0
2356,entity_00002356,irina gorshkova,81,arm,1,1,0,0
1791,entity_00001791,fayrbyrd menejment spy,76,arm,1,1,1,0
4152,entity_00004152,star dast pby,74,arm,1,1,3,0
1783,entity_00001783,evgenia vasileva,71,arm,1,1,0,0


In [9]:
# ============================================================
# SAMPLE OF FINAL DEDUPLICATED RECORDS
# ============================================================

display(
    deduplicated_sample[
        [
            "record_id",
            "country",
            "party_name",
            "name_latin",
            "entity_id",
            "cluster_size",
            "canonical_name",
        ]
    ]
    .sample(20, random_state=42)
)

,record_id,country,party_name,name_latin,entity_id,cluster_size,canonical_name
57281,mng_rec_bd30361cfe68a22159bf0a92a0ba8737,mng,мягмар урангоо,myagmar urangoo,single_2957330,1,myagmar urangoo
16458,kgz_rec_1a91019a563eaa425aa1001775c43c18,kgz,Мамытов Акжигит Атаярович,mamytov akzhigit atayarovich,single_1661852,1,mamytov akzhigit atayarovich
15165,azb_rec_8b9e0f18ecdc16a2c6db71613f1330cf,azb,GEORGE SİM GORDON,george si m gordon,single_606021,1,george si m gordon
41746,mng_rec_097a0387bf5acc14e4634b3a39ccc8b1,mng,Төмөрхуяг Лувсандолгор,tomorhuyag luvsandolgor,single_3386230,1,tomorhuyag luvsandolgor
7110,kgz_rec_ae9e8ddc93d6aa8da36bb9e71c025798,kgz,Огонбаев Радж Кашкарович,ogonbaev radzh kashkarovich,entity_00003438,2,ogonbaev radzh kashkarovich
32013,mng_rec_0d03065a4218bc07ce3baa8b66e9cafa,mng,чиндэгсүрэн анхзаяа,chindegsuren anhzayaa,single_2803098,1,chindegsuren anhzayaa
81584,azb_rec_4f95f1905db5be3de6ca28c932d09f3b,azb,UMUDOV ADİL MƏHYƏDDİN OĞLU,umudov adi l mehyeddi n og lu,single_479435,1,umudov adi l mehyeddi n og lu
30640,mng_rec_02967de334b46f1a653f0ed65dccc232,mng,бор батмөнх,bor batmonh,single_2794767,1,bor batmonh
2225,tjk_rec_085ea48fadfd3e05d9f05b4c29f58788,tjk,Давлятов Исмонали Абдурахимович,davlyatov ismonali abdurahimovich,single_3681709,1,davlyatov ismonali abdurahimovich
30835,mng_rec_5ede48b1f6122cea30bd4f980b7b8e42,mng,Есенкелди Сарымсах,esenkeldi sarymsah,entity_00001771,2,esenkeldi sarymsah


In [10]:
# ============================================================
# SAVE FINAL EXPORTS
# ============================================================

deduplicated_sample.to_parquet(
    "../data/processed/deduplicated_sample.parquet",
    index=False,
)

canonical_entities.to_parquet(
    "../data/processed/canonical_entities_full_sample.parquet",
    index=False,
)

print("Saved:")
print("../data/processed/deduplicated_sample.parquet")
print("../data/processed/canonical_entities_full_sample.parquet")

Saved:
../data/processed/deduplicated_sample.parquet
../data/processed/canonical_entities_full_sample.parquet
